# EDA tầng Bronze — OutbreakSignal DE

**Chạy trên Google Colab, không cần cài gì trên máy.** Notebook tự tải dữ liệu từ
đúng các nguồn mà pipeline dùng, dựng tầng Bronze ngay trong Colab, rồi phân tích.

Mục tiêu: trả lời ba câu hỏi trước khi thiết kế tầng Silver.

1. Dữ liệu phủ tới **đơn vị địa lý nào** — quốc gia, tỉnh, hay nhỏ hơn?
2. **Giá trị** trông ra sao — thiếu, âm, trùng lặp, lệch phân bố?
3. Ba nguồn có **ghép được với nhau** không?

> ⚠️ EDA chỉ **đọc** Bronze, không ghi đè hay sửa gì. Nguyên tắc "Bronze giữ
> nguyên trạng nguồn" vẫn được tôn trọng — không phép biến đổi nào được ghi
> ngược trở lại.

Thời gian chạy: khoảng 5–7 phút (phần lớn là cài PySpark và tải dữ liệu).


---
## Phần 0 — Cài đặt

Bản `pyspark` phải là **3.5.9**: các bản 3.5.x cũ hơn dính
[SPARK-53759](https://issues.apache.org/jira/browse/SPARK-53759).


In [ ]:
!pip install -q pyspark==3.5.9 delta-spark==3.3.3
print('Xong. Chạy tiếp cell sau.')


In [ ]:
import json
import zipfile
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import requests
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Bảng màu đã kiểm định (tương phản + mù màu). Ba slot đầu dùng được cho mọi
# loại biểu đồ; đừng tự đổi sang màu khác.
BLUE, ORANGE, AQUA = '#2a78d6', '#eb6834', '#1baf7a'
INK, MUTED = '#0b0b0b', '#52514e'

def style_axes(ax, title, xlabel=''):
    """Lưới mờ, bỏ khung, chữ dùng màu chữ chứ không dùng màu series."""
    ax.set_title(title, color=INK, fontsize=13, pad=14, loc='left')
    ax.set_xlabel(xlabel, color=MUTED, fontsize=10)
    ax.tick_params(colors=MUTED, labelsize=10, length=0)
    for side in ('top', 'right', 'left', 'bottom'):
        ax.spines[side].set_visible(False)
    ax.grid(axis='x', color='#e5e5e2', linewidth=0.8)
    ax.set_axisbelow(True)

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
print('Import xong.')


In [ ]:
builder = (
    SparkSession.builder.appName('eda_bronze_colab')
    .master('local[*]')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog',
            'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.sql.session.timeZone', 'UTC')
    .config('spark.ui.showConsoleProgress', 'false')
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark', spark.version)


---
## Phần 1 — Dựng Bronze trong Colab

Cùng URL, cùng quy tắc với pipeline thật (`configs/sources.yaml`):
`inferSchema=False`, không đổi tên cột, không lọc, không ép kiểu. Chỉ thêm cột
truy vết.

Nếu đã có sẵn `data/bronze/` từ máy, xem **Phụ lục** ở cuối để upload và bỏ qua
phần này.


In [ ]:
SOURCES = {
    'opendengue': {
        'url': ('https://github.com/OpenDengue/master-repo/raw/main/data/'
                'releases/V1.3/National_extract_V1_3.zip'),
    },
    'news_rss': {
        'url': 'https://news.google.com/rss/search',
        'params': {'q': 'dengue Southeast Asia', 'hl': 'en',
                   'gl': 'US', 'ceid': 'US:en'},
    },
    'sg_nea': {
        'dataset_id': 'd_dbfabf16158d1b0e1c420627c0819168',
        'url': ('https://api-open.data.gov.sg/v1/public/api/datasets/'
                '{dataset_id}/poll-download'),
    },
}

LANDING = Path('/content/data/landing')
BRONZE = Path('/content/data/bronze')
TODAY = datetime.now(timezone.utc).strftime('%Y-%m-%d')
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')

def landing_dir(source):
    path = LANDING / source / TODAY
    path.mkdir(parents=True, exist_ok=True)
    return path

def write_bronze(df, source):
    """Thêm cột truy vết rồi ghi Delta. Không đụng vào cột gốc."""
    enriched = (df
        .withColumn('_source', F.lit(source))
        .withColumn('_ingested_at', F.lit(datetime.now(timezone.utc).isoformat()))
        .withColumn('ingestion_date', F.lit(TODAY)))
    (enriched.write.format('delta').mode('overwrite')
        .partitionBy('ingestion_date').save(str(BRONZE / source)))
    return enriched

print('Cấu hình xong. ingestion_date =', TODAY)


In [ ]:
# --- OpenDengue: zip -> CSV -> Bronze ---
dest = landing_dir('opendengue')
resp = requests.get(SOURCES['opendengue']['url'], timeout=120)
resp.raise_for_status()
zip_path = dest / 'National_extract_V1_3.zip'
zip_path.write_bytes(resp.content)

with zipfile.ZipFile(zip_path) as archive:
    csv_name = [n for n in archive.namelist() if n.lower().endswith('.csv')][0]
    archive.extract(csv_name, dest)

# inferSchema=False -> tất cả là string. CỐ Ý: Bronze không ép kiểu.
od = write_bronze(
    spark.read.csv(str(dest / '*.csv'), header=True, inferSchema=False),
    'opendengue')
print(f'opendengue: {od.count():,} dòng, {len(od.columns)} cột')


In [ ]:
# --- Google News RSS: XML -> JSONL -> Bronze ---
cfg = SOURCES['news_rss']
dest = landing_dir('news_rss')
resp = requests.get(cfg['url'], params=cfg['params'],
                    headers={'User-Agent': 'OutbreakSignalDE/0.1'}, timeout=60)
resp.raise_for_status()
(dest / f'google_news_{STAMP}.xml').write_bytes(resp.content)

FIELDS = ('title', 'link', 'pubDate', 'source', 'description')
items = [{f: (it.findtext(f) or '').strip() for f in FIELDS}
         for it in ET.fromstring(resp.content).findall('.//item')]

with open(dest / f'google_news_{STAMP}.jsonl', 'w', encoding='utf-8') as handle:
    for item in items:
        handle.write(json.dumps(item, ensure_ascii=False) + '\n')

news = write_bronze(spark.read.json(str(dest / '*.jsonl')), 'news_rss')
print(f'news_rss: {news.count():,} dòng, {len(news.columns)} cột')


In [ ]:
# --- Singapore NEA: poll-download 2 chặng -> GeoJSON -> Bronze ---
cfg = SOURCES['sg_nea']
envelope = requests.get(cfg['url'].format(dataset_id=cfg['dataset_id']),
                        timeout=30).json()
assert envelope.get('code') == 0, envelope.get('errorMsg')
geojson = requests.get(envelope['data']['url'], timeout=30).json()

dest = landing_dir('sg_nea')
(dest / f'clusters_{STAMP}.json').write_text(
    json.dumps(geojson, ensure_ascii=False), encoding='utf-8')

rows = []
for feature in geojson['features']:
    props = feature.get('properties', {})
    locality = props.get('LOCALITY')
    rows.append({
        'object_id': str(props.get('OBJECTID')),
        'locality': locality.strip() if locality else None,
        'case_count': props.get('CASE_SIZE'),
        'cluster_updated_at_raw': props.get('FMEL_UPD_D'),
        'inc_crc': props.get('INC_CRC'),
        'polygon_geojson': json.dumps(feature.get('geometry'), sort_keys=True),
    })

# coalesce(1): vài chục dòng không có lý do gì chia ra nhiều phân vùng.
sg = write_bronze(spark.createDataFrame(rows).coalesce(1), 'sg_nea')
print(f'sg_nea: {sg.count():,} dòng, {len(sg.columns)} cột')


---
## Phần 2 — OpenDengue

### 2.1 Dữ liệu phủ tới đơn vị địa lý nào?

Đây là câu hỏi quan trọng nhất. Bảng có ba cột `adm_0_name` / `adm_1_name` /
`adm_2_name` (quốc gia / tỉnh / huyện) nên **trông như** có dữ liệu cấp tỉnh.


In [ ]:
print('--- Số giá trị phân biệt của từng cấp hành chính ---')
od.select(
    F.countDistinct('adm_0_name').alias('so_quoc_gia'),
    F.countDistinct('adm_1_name').alias('so_tinh'),
    F.countDistinct('adm_2_name').alias('so_huyen'),
).show()

print('--- adm_1_name / adm_2_name thực sự chứa gì? ---')
od.select('adm_1_name', 'adm_2_name').distinct().show(5, truncate=False)

print('--- S_res: độ phân giải không gian nguồn tự khai báo ---')
od.groupBy('S_res').count().show(truncate=False)


**Kết luận 2.1 — file này KHÔNG có dữ liệu cấp tỉnh.**

`adm_1_name` và `adm_2_name` chứa đúng một giá trị: chuỗi `"NA"`. Cột tồn tại
nhưng rỗng nghĩa. `S_res = Admin0` cho 100% số dòng — nguồn tự khai là dữ liệu
cấp quốc gia.

Đây là bản `National_extract`. OpenDengue có phát hành thêm **spatial extract**
chứa dữ liệu dưới cấp quốc gia — cần cấp tỉnh thì phải tải file khác.

> **Ảnh hưởng tới bài toán:** "khoanh vùng nguy cơ" ở mức tỉnh **không làm được**
> với nguồn hiện tại. Hoặc đổi sang spatial extract, hoặc hạ mục tiêu xuống mức
> quốc gia. Phải quyết trước khi thiết kế Silver.


### 2.2 `T_res` — độ phân giải thời gian không đồng nhất

Cột `T_res` cho biết mỗi dòng là số liệu của một **tuần**, một **tháng**, hay cả
một **năm**. Ba loại nằm lẫn trong cùng một bảng.

Câu hỏi quan trọng: chúng có **chồng lên nhau** không? Nếu cùng một nước trong
cùng một năm vừa có dòng tuần vừa có dòng năm thì `SUM` sẽ đếm trùng.


In [ ]:
od.groupBy('T_res').count().orderBy(F.desc('count')).show(truncate=False)

print('--- Mỗi cặp (nước, năm) có mấy loại T_res? ---')
combo = (od.groupBy('adm_0_name', 'Year')
           .agg(F.countDistinct('T_res').alias('so_loai_T_res')))
(combo.groupBy('so_loai_T_res').agg(F.count('*').alias('so_cap_nuoc_nam'))
      .orderBy('so_loai_T_res').show(truncate=False))


**Kết luận 2.2 — KHÔNG bị đếm trùng, nhưng độ phân giải không đồng nhất.**

Mọi cặp (nước, năm) đều chỉ có **đúng một** loại `T_res`. Nghĩa là các độ phân
giải **không chồng lên nhau**, và `SUM(dengue_total)` không đếm trùng. Đây là tin
tốt — nhưng phải kiểm tra mới biết, đừng giả định.

Vấn đề thật nằm ở chỗ khác: **cùng một nước đổi độ phân giải theo thời gian.**
Cell dưới cho thấy rõ với Việt Nam.


In [ ]:
(od.filter(F.col('adm_0_name') == 'VIET NAM')
   .groupBy('Year').pivot('T_res', ['Week', 'Month', 'Year']).count()
   .orderBy('Year').show(70, truncate=False))


Việt Nam: chỉ có số liệu **năm** cho giai đoạn 1960–1994, chuyển sang **tháng**
khoảng 1995–2015, và chỉ từ **2016** mới có số liệu **tuần**.

> **Ảnh hưởng tới Silver:** không thể dựng một chuỗi thời gian theo tuần cho toàn
> bộ lịch sử. Ba lựa chọn:
>
> 1. Chỉ lấy giai đoạn có `Week` cho mọi nước đích (mất phần lịch sử dài).
> 2. Gộp tất cả lên grain thô hơn — tháng hoặc năm (mất độ mịn gần đây).
> 3. Giữ `T_res` thành một cột trong Silver, để tầng sau tự quyết.
>
> Với bài toán *cảnh báo sớm* thì dữ liệu gần đây quan trọng hơn lịch sử xa, nên
> hướng 1 hợp lý nhất cho MVP.


### 2.3 Phủ Đông Nam Á — và một cái bẫy đặt tên

Tên nước trong dataset theo chuẩn UN, **không** phải cách viết thông thường.


In [ ]:
print('--- Viết "VIETNAM" liền (cách spikes/ đang lọc) ---')
print('số dòng:', od.filter(F.col('adm_0_name') == 'VIETNAM').count())

print()
print('--- Nguồn thực sự viết thế nào? ---')
(od.filter(F.col('adm_0_name').rlike('(?i)viet'))
   .select('adm_0_name').distinct().show(truncate=False))


> 🐛 **Lỗi thật trong code hiện tại.** `spikes/test_opendengue.py` lọc bằng
> `"VIETNAM"` viết liền, trong khi nguồn ghi `"VIET NAM"` có dấu cách. Việt Nam bị
> loại âm thầm — đó là lý do ghi chú tiến độ nói "10 nước" thay vì 11. Cần sửa
> trước khi viết báo cáo.


In [ ]:
SEA = ['VIET NAM', 'THAILAND', 'INDONESIA', 'PHILIPPINES', 'MALAYSIA',
       'MYANMAR', 'CAMBODIA', "LAO PEOPLE'S DEMOCRATIC REPUBLIC",
       'SINGAPORE', 'BRUNEI DARUSSALAM', 'TIMOR-LESTE']

sea = od.filter(F.col('adm_0_name').isin(SEA))
print(f'11 nước SEA: {sea.count():,} dòng / tổng {od.count():,} dòng toàn cầu')

(sea.groupBy('adm_0_name')
    .agg(F.count('*').alias('so_dong'),
         F.min('calendar_start_date').alias('tu_ngay'),
         F.max('calendar_end_date').alias('den_ngay'))
    .orderBy(F.desc('so_dong')).show(20, truncate=False))


#### Biểu đồ — số dòng theo nước, tách theo độ phân giải thời gian

Cố ý vẽ **số dòng** chứ không vẽ tổng số ca: tổng số ca bị đếm trùng vì lý do ở
mục 2.2.


In [ ]:
pivot = (sea.groupBy('adm_0_name').pivot('T_res', ['Week', 'Month', 'Year'])
            .count().fillna(0).toPandas().set_index('adm_0_name'))
pivot['tong'] = pivot.sum(axis=1)
pivot = pivot.sort_values('tong')

fig, ax = plt.subplots(figsize=(9, 5.5))
left = [0] * len(pivot)
for col, color in zip(['Week', 'Month', 'Year'], [BLUE, ORANGE, AQUA]):
    ax.barh(pivot.index, pivot[col], left=left, height=0.62, color=color,
            label=col, edgecolor='white', linewidth=2)
    left = [a + b for a, b in zip(left, pivot[col])]

# Nhãn giá trị hiện rõ: bắt buộc, vì có màu tương phản thấp trên nền sáng.
for y, total in enumerate(pivot['tong']):
    ax.text(total + 8, y, f'{int(total)}', va='center', color=MUTED, fontsize=9)

style_axes(ax, 'Số dòng OpenDengue theo nước, tách theo độ phân giải thời gian',
           'số dòng')
ax.legend(frameon=False, loc='lower right', fontsize=10, labelcolor=MUTED)
ax.set_xlim(0, pivot['tong'].max() * 1.12)
plt.tight_layout()
plt.show()


### 2.4 Chất lượng giá trị `dengue_total`


In [ ]:
od.select(
    F.count('*').alias('tong_dong'),
    F.count(F.when(F.col('dengue_total').isNull(), 1)).alias('null'),
    F.count(F.when(F.trim('dengue_total') == '', 1)).alias('chuoi_rong'),
    F.count(F.when(F.col('dengue_total').cast('double') < 0, 1)).alias('gia_tri_am'),
    F.count(F.when(F.col('dengue_total').isNotNull() &
                   F.col('dengue_total').cast('double').isNull(), 1)
            ).alias('khong_ep_kieu_duoc'),
).show()


---
## Phần 3 — Google News RSS

### 3.1 Trùng lặp giữa các lần fetch


In [ ]:
print('tổng số dòng    :', news.count())
print('số link duy nhất:', news.select('link').distinct().count())

print()
print('--- một bài xuất hiện mấy lần? ---')
(news.groupBy('link').count()
     .withColumnRenamed('count', 'so_lan_xuat_hien')
     .groupBy('so_lan_xuat_hien').agg(F.count('*').alias('so_bai'))
     .orderBy('so_lan_xuat_hien').show(truncate=False))


Bronze giữ mọi lần fetch là **đúng thiết kế** — mỗi lần fetch là một lần quan sát
riêng. Nhưng con số này định lượng việc Silver phải làm: **dedup theo `link`**.


### 3.2 Có suy ra được quốc gia không?

RSS không có trường địa điểm nào. Cách duy nhất là dò từ khoá trong tiêu đề — đo
xem tỉ lệ dò được là bao nhiêu.


In [ ]:
print('các cột hiện có:',
      [c for c in news.columns if not c.startswith(('_', 'ingestion'))])

KEYWORDS = {
    'Vietnam': r'viet ?nam', 'Thailand': r'thailand|thai\b',
    'Indonesia': r'indonesia', 'Philippines': r'philippine',
    'Malaysia': r'malaysia', 'Singapore': r'singapore',
    'Myanmar': r'myanmar|burma', 'Cambodia': r'cambodia',
    'Laos': r'laos|lao pdr', 'Brunei': r'brunei', 'Timor-Leste': r'timor',
}

haystack = F.lower(F.concat_ws(' ', F.col('title'), F.col('description')))
labelled = news.select('title', 'link', haystack.alias('text')).distinct()

matched = F.lit(False)
for country, pattern in KEYWORDS.items():
    matched = matched | F.col('text').rlike(pattern)
labelled = labelled.withColumn('co_quoc_gia', matched)

total = labelled.count()
hit = labelled.filter('co_quoc_gia').count()
print(f'\nDò được quốc gia: {hit}/{total} bài  ({hit / total:.0%})')
print(f'Không dò được   : {total - hit}/{total} bài  ({1 - hit / total:.0%})')

print('\n--- ví dụ bài KHÔNG dò được quốc gia ---')
labelled.filter(~F.col('co_quoc_gia')).select('title').show(8, truncate=90)


**Kết luận Phần 3.** Tỉ lệ dò được chính là **giới hạn trên** độ chính xác của
nguồn tin tức khi gán về quốc gia. Con số này nên đưa thẳng vào báo cáo — nó cho
thấy nhóm đã *đo* giới hạn của nguồn chứ không giả định nguồn hoàn hảo.


---
## Phần 4 — Singapore NEA

### 4.1 Phân bố số ca theo cụm


In [ ]:
(sg.select('locality', 'case_count', 'cluster_updated_at_raw')
   .orderBy(F.desc('case_count')).show(20, truncate=55))

sg.select(
    F.count('*').alias('so_cum'),
    F.sum('case_count').alias('tong_ca'),
    F.min('case_count').alias('nho_nhat'),
    F.max('case_count').alias('lon_nhat'),
    F.round(F.avg('case_count'), 1).alias('trung_binh'),
    F.expr('percentile_approx(case_count, 0.5)').alias('trung_vi'),
).show()


In [ ]:
clusters = sg.select('locality', 'case_count').toPandas().sort_values('case_count')
clusters['nhan'] = clusters['locality'].str.slice(0, 38)

fig, ax = plt.subplots(figsize=(9, max(4, 0.34 * len(clusters))))
ax.barh(clusters['nhan'], clusters['case_count'], height=0.62, color=BLUE)
for y, value in enumerate(clusters['case_count']):
    ax.text(value + clusters['case_count'].max() * 0.012, y, str(int(value)),
            va='center', color=MUTED, fontsize=9)

style_axes(ax, 'Số ca theo cụm dịch — Singapore NEA', 'số ca')
ax.set_xlim(0, clusters['case_count'].max() * 1.12)
plt.tight_layout()
plt.show()


**Kết luận 4.1 — phân bố lệch cực mạnh, trung bình là con số gây hiểu nhầm.**

Một cụm thường chiếm phần lớn tổng số ca, các cụm còn lại chỉ 2–4 ca. Giá trị
`trung_binh` không mô tả đúng cụm nào cả — hãy so nó với `trung_vi` ở trên.

> **Ảnh hưởng tới trang cảnh báo:** chỉ số rủi ro **không được dùng trung bình**.
> Dùng trung vị, hoặc chính giá trị lớn nhất. Đây cũng là đặc trưng dịch tễ hợp lý
> — ổ dịch lớn thì hiếm, ổ nhỏ thì nhiều.


### 4.2 Nguồn publish bao lâu một lần?

Khác với "chúng ta poll bao lâu một lần" — đây mới là con số quyết định nhịp chạy.


In [ ]:
(sg.groupBy('cluster_updated_at_raw')
   .agg(F.count('*').alias('so_cum'))
   .withColumn('doc_duoc',
               F.to_timestamp('cluster_updated_at_raw', 'yyyyMMddHHmmss'))
   .orderBy('cluster_updated_at_raw').show(truncate=False))


`FMEL_UPD_D` không mang thông tin múi giờ. Nếu là UTC thì 15:00 UTC = 23:00 giờ
Singapore — giờ publish lạ với cơ quan nhà nước. Nếu là SGT thì là đầu giờ chiều,
hợp lý hơn nhiều. Bronze giữ nguyên chuỗi gốc để Silver quyết định sau khi có
bằng chứng chắc chắn.


---
## Phần 5 — Ghép ba nguồn: vấn đề thật sự

Đây là phát hiện quan trọng nhất của cả notebook.


In [ ]:
print('ĐƠN VỊ ĐỊA LÝ CỦA TỪNG NGUỒN')
print('=' * 68)

print('\n[opendengue] cấp quốc gia:')
od.select('adm_0_name').distinct().show(3, truncate=False)

print('[news_rss] không có trường địa điểm nào:')
print('  ', [c for c in news.columns if not c.startswith(('_', 'ingestion'))])

print('\n[sg_nea] cấp cụm dân cư, là chuỗi tự do:')
sg.select('locality').show(3, truncate=False)


| Nguồn | Đơn vị địa lý | Ghép được không? |
|---|---|---|
| OpenDengue | Quốc gia (`VIET NAM`) | — |
| Google News RSS | **Không có** | phải dò từ tiêu đề |
| Singapore NEA | Cụm phố (`Bt Batok St 21 (Blk 207...)`) | chuỗi tự do, không phải đơn vị hành chính |

**Ba nguồn nằm ở ba cấp địa lý khác nhau và không có khoá chung.** Singapore là chỗ
duy nhất chồng lấn, mà ngay ở đó OpenDengue cho một con số toàn quốc còn NEA cho
polygon từng cụm phố.

### Điều này quyết định thiết kế Silver

Không thể join ba nguồn thành một bảng phẳng. Hai hướng khả dĩ:

1. **Hạ tất cả về cấp quốc gia.** Mất thông tin polygon của NEA, nhưng ba nguồn xếp
   chung được một bảng. Đơn giản, làm kịp.
2. **Giữ nhiều cấp trong một bảng**, có cột `admin_level` để phân biệt. Giữ được dữ
   liệu chi tiết nhưng mọi truy vấn sau đều phải lọc theo cấp.

Gợi ý cho MVP: **hướng 1**, và ghi rõ trong báo cáo là đã cân nhắc hướng 2.


---
## Phần 6 — Tóm tắt phát hiện

| # | Phát hiện | Ảnh hưởng |
|---|---|---|
| 1 | OpenDengue `National_extract` **không có cấp tỉnh** (`adm_1/adm_2 = "NA"`, `S_res = Admin0`) | Không khoanh vùng cấp tỉnh được; phải đổi sang spatial extract hoặc hạ mục tiêu |
| 2 | `T_res` trộn Week/Month/Year, nhưng mỗi (nước, năm) chỉ có 1 loại — **không đếm trùng** | Grain đổi theo thời gian: VN chỉ có `Week` từ 2016. Không dựng được chuỗi tuần cho cả lịch sử |
| 3 | Nguồn ghi `VIET NAM` có dấu cách, code lọc `VIETNAM` | Việt Nam bị loại âm thầm — **lỗi cần sửa** |
| 4 | News RSS trùng nhiều giữa các lần fetch | Silver phải dedup theo `link` |
| 5 | News RSS không có trường địa điểm | Phải dò từ khoá; tỉ lệ dò được là giới hạn trên của độ chính xác |
| 6 | NEA phân bố số ca lệch cực mạnh | Chỉ số rủi ro không được dùng trung bình |
| 7 | Ba nguồn ở ba cấp địa lý, không có khoá chung | Quyết định lớn nhất của thiết kế Silver |

### Việc tiếp theo

- Sửa bộ lọc `VIETNAM` → `VIET NAM` trong `spikes/test_opendengue.py`
- Quyết định cấp địa lý đích của Silver (gợi ý: quốc gia)
- Quyết định grain thời gian đích + giai đoạn áp dụng (gợi ý: `Week`, từ 2016)
- Chụp lại các biểu đồ và bảng trên để đưa vào báo cáo


---
## Phụ lục — dùng dữ liệu có sẵn thay vì tải mới

Nếu đã chạy pipeline trên máy và muốn EDA đúng bộ dữ liệu đó, nén `data/bronze/`
rồi upload:

```python
from google.colab import files
files.upload()                       # chọn bronze.zip
!unzip -q bronze.zip -d /content/data/
```

Rồi đọc lại và bỏ qua Phần 1:

```python
od   = spark.read.format('delta').load('/content/data/bronze/opendengue')
news = spark.read.format('delta').load('/content/data/bronze/news_rss')
sg   = spark.read.format('delta').load('/content/data/bronze/sg_nea')
```
